本文基于 `langchain==1.3.0`，重点讲 6 件事：工具在 Agent 中扮演什么角色、如何写最小自定义工具、如何处理复杂参数、怎么使用预定义工具、何时对工具做二次封装，以及真实项目里该如何取舍。

## 一、工具在 Agent 中的角色

在 LangChain 里，`Tool` 可以理解为：**把模型的纯文本生成能力扩展成“可调用能力”。**

它本质上是一个带有明确输入、输出约束的可调用对象。交给模型后，模型会根据上下文决定：
- 什么时候要调用工具
- 应该调用哪个工具
- 调用时要传什么参数

即，用户把问题交给 Agent，模型决定是否调用工具，工具执行后再把结果交还给模型生成答案。

用代码来说明。

In [ ]:
# 加载环境变量，并初始化本篇会复用的模型
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(
    model="qwen3.5-122b-a10b",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0,
)


## 二、从零写一个最小自定义工具

最常见的起点是 `@tool`。

先记住 3 个原则：
- 函数名默认会成为工具名，所以尽量使用 `snake_case`
- 类型注解会参与生成输入 schema，不要省略
- docstring 会成为工具说明，直接影响模型是否会正确调用它

下面先看最小示例。

前提：底层聊天模型需要支持 tool calling。


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

# 用 @tool 把普通函数注册成 LangChain 工具
@tool
def get_weather(location: str) -> str:
    """获取指定城市的模拟天气信息。"""
    # 这里用静态数据模拟外部天气服务
    mock_data = {
        "上海": "小雨，22°C，体感偏凉，建议带伞。",
        "北京": "晴天，28°C，紫外线较强，建议做好防晒。",
        "杭州": "多云，24°C，适合出行。",
    }
    return mock_data.get(location, f"暂时没有查询到{location}的天气数据。")

# 把工具注册给 Agent，后续由模型决定是否调用
agent = create_agent(
    model=model,
    tools=[get_weather],
)


In [ ]:
# 调用 Agent
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "我明天要去上海出差，先帮我查一下天气，再告诉我要不要带伞。"
        }
    ]
})

print(response["messages"][-1].content)


这个例子真正值得看的是调用链路：
- 用户并没有直接点名 `get_weather`
- Agent 会把工具列表交给模型
- 模型自己判断：这个问题需要借助天气工具
- 工具返回结果后，模型再基于工具结果组织最终回复

In [ ]:
# 工具本身还带有可供模型读取的元信息
print(get_weather.name)
print(get_weather.description)
print(get_weather.args)

## 三、复杂参数与结构化输入

当工具输入开始复杂时，推荐用 `Pydantic` 定义 `args_schema`。

直接收益有两个：
- 参数结构更清晰
- 字段描述会直接帮助模型理解每个参数的含义


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

# 用 Pydantic 明确声明工具的输入结构
class TravelAssistantInput(BaseModel):
    city: str = Field(description="要查询的城市，例如上海、北京")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="温度单位"
    )
    include_advice: bool = Field(
        default=True,
        description="是否返回出行建议"
    )

# 当参数变多时，用 args_schema 比裸函数参数更稳定
@tool(args_schema=TravelAssistantInput)
def get_travel_weather(
    city: str,
    units: Literal["celsius", "fahrenheit"] = "celsius",
    include_advice: bool = True,
) -> dict:
    """查询某个城市的天气，并返回结构化的差旅建议。"""
    weather_data = {
        "上海": {"temperature_c": 22, "condition": "小雨", "advice": "建议带伞，外套可以备一件。"},
        "北京": {"temperature_c": 28, "condition": "晴天", "advice": "注意防晒，白天较热。"},
        "深圳": {"temperature_c": 30, "condition": "闷热", "advice": "建议轻装出行，并准备补水。"},
    }

    data = weather_data.get(city, {"temperature_c": 25, "condition": "未知", "advice": "暂无更多建议。"})
    # 根据用户要求切换温度单位
    temperature = data["temperature_c"] if units == "celsius" else round(data["temperature_c"] * 9 / 5 + 32)

    # 返回结构化结果，方便模型继续组织答案
    result = {
        "city": city,
        "temperature": temperature,
        "units": units,
        "condition": data["condition"],
    }
    if include_advice:
        result["advice"] = data["advice"]
    return result

advanced_agent = create_agent(
    model=model,
    tools=[get_travel_weather],
)


In [ ]:
# 让模型自己填入 city、units、include_advice 这 3 个参数
response = advanced_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "帮我看一下北京的天气，用华氏度表示，并给我一个简短的出行建议。"
        }
    ]
})

print(response["messages"][-1].content)


这个示例要注意两点：
- 工具可以返回 `dict`，不一定只能返回字符串；结构化结果更适合模型的后续推理。
- `Field(description=...)` 不是可有可无的注释，它会直接影响模型填参是否准确。
- 更完整地说，工具还可以返回 `Command` 来更新状态；不过本文先聚焦最常见的字符串和对象返回。

小结：
- 工具名尽量用 `snake_case`，不要带空格和奇怪符号
- docstring 要说清楚“这个工具是干什么的”，而不是只写一句空泛描述
- 参数类型注解不要省略，因为它会决定输入 schema
- 参数一多，就优先用 `args_schema` 明确定义
- 返回文本适合直接给模型阅读，返回对象适合让模型基于字段继续分析

![LangChain Tools 架构图](assets/两种自定义工具写法的对比.jpg)

## 四、直接使用预定义工具

预定义工具是社区或官方已经封装好的工具。下面用 `TavilySearch` 做一个最小示例。

使用 Tavily 前，只需要完成 3 步：
1. 在 <https://app.tavily.com> 获取 `TAVILY_API_KEY`
2. 安装依赖：`uv add langchain-tavily`（或 `pip install langchain-tavily`）
3. 在 `.env` 中写入：`TAVILY_API_KEY=tvly-your-real-api-key`


In [ ]:
from langchain.agents import create_agent

# 没安装集成包时，先优雅降级，避免 notebook 直接报错
try:
    from langchain_tavily import TavilySearch
except ImportError:
    TavilySearch = None

if TavilySearch is not None:
    # 直接使用预定义搜索工具
    tavily_search = TavilySearch(max_results=3, topic="general")
    web_agent = create_agent(model=model, tools=[tavily_search])


In [ ]:
import json
from langchain.messages import ToolMessage

prompt = "搜索并用一句话告诉我 LangChain 自定义工具有什么用途"

if TavilySearch is not None and os.environ.get("TAVILY_API_KEY"):
    # 这里直接让 Agent 使用预定义搜索工具
    response_6 = web_agent.invoke({
        "messages": [{"role": "user", "content": prompt}]
    })
    print(response_6["messages"][-1].content)
    
    # 从response里确认工具的搜索来源
    for message in response_6["messages"]:
        if isinstance(message, ToolMessage):
            data = json.loads(message.content)
            for item in data.get("results", []):
                print("搜索来源：",item.get("url"))
else:
    print("请先安装 langchain-tavily，并配置 TAVILY_API_KEY。")

## 五、对预定义工具再包一层

预定义工具接入很快，但默认配置往往偏通用。

如果你希望它更贴近当前场景，可以再包一层。下面只把搜索范围固定到 `docs.langchain.com`。


In [ ]:
from langchain.tools import tool
import json

if TavilySearch is not None:
    # 复用 Tavily 的底层能力，但改造成更聚焦的业务工具
    raw_tavily_search = TavilySearch(max_results=3, topic="general")

    @tool
    def search_langchain_docs(query: str) -> str:
        """搜索 LangChain 官方文档，并返回相关链接。"""
        # 把搜索范围限制到官方文档站点
        result = raw_tavily_search.invoke({
            "query": query,
            "include_domains": ["docs.langchain.com"],
        })

        if isinstance(result, str):
            try:
                result = json.loads(result)
            except json.JSONDecodeError:
                return result or "没有找到相关文档结果。"

        # 统一整理输出，避免把原始结果整包丢给模型
        lines = []
        for item in result.get("results", [])[:3]:
            title = item.get("title", "无标题")
            url = item.get("url", "")
            lines.append(f"{title}\n{url}")

        return "\n\n".join(lines) if lines else "没有找到相关文档结果。"

    wrapped_agent = create_agent(model=model, tools=[search_langchain_docs])


In [ ]:
if TavilySearch is not None and os.environ.get("TAVILY_API_KEY"):
    # 对比上一个示例：这次调用的是我们包装过的工具
    response_7 = wrapped_agent.invoke({
        "messages": [{"role": "user", "content": prompt}]
    })
    print(response_7["messages"][-1].content)

    # 从response里确认工具的搜索来源
    for message in response_7["messages"]:
        if isinstance(message, ToolMessage):
            for line in message.content.splitlines():
                if line.startswith("http"):
                    print("搜索来源：",line)
    
else:
    print("请先安装 langchain-tavily，并配置 TAVILY_API_KEY。")

两次提示词完全相同，差别主要在工具约束方式：直接用预定义工具时，默认搜索范围更宽；包成自定义工具后，我们把能力边界收窄到了 `docs.langchain.com`，所以结果会更聚焦、更可控。

## 六、真实项目里怎么选工具

如果只是学习机制，建议先写自定义工具，因为它最容易看清调用链。进入真实项目后，可以按下面的思路判断：
![LangChain Tools 架构图](assets/工具选型决策树.jpg)

还有一个常见误区：工具不是越多越好。工具一多，模型选择成本和误调用概率都会上升。更实用的做法是：
- 只给当前任务真正需要的工具
- 工具说明尽量清晰，减少语义重叠
- 能合并的工具不要拆得过碎
- 能限制作用域的工具，尽量限制作用域